# MODULE 41 — OBJECT DETECTION ARCHITECTURES

## Section 3: Modern Detector Awareness

<badge>Must Know Conceptually</badge> **Modern Computer Vision**

Now let's move beyond traditional object detection architectures and understand the concepts behind modern detectors.

So far, we have studied:

- Two-stage detectors: R-CNN, Fast R-CNN, Faster R-CNN.
- Single-stage detectors: SSD, RetinaNet, YOLO.

Modern object detection has evolved through improvements in feature extraction, bounding-box prediction and attention mechanisms.

In this section, we'll study four important concepts:

1. Feature Pyramids
2. Anchor-based vs Anchor-free Detection
3. Transformer-based Object Detection
4. DETR — Detection Transformer

---

# 1. Feature Pyramids

<badge>Must Know</badge>

## 1.1 Why do we need Feature Pyramids?

Imagine an image containing three objects:

- A small bird sitting on a tree.
- A medium-sized car.
- A large bus.

![Objects of different sizes in an image](https://images.unsplash.com/photo-1501706362039-c6e13d8e6e44?w=1200)

The problem is that objects appear in different sizes.

A small object might occupy only a few pixels, while a large object can occupy a significant portion of the image.

Can a CNN detect all these objects equally well using only one feature map?

Not necessarily.

Let's understand why.

## 1.2 CNN Feature Maps at Different Layers

When an image passes through a CNN, it generates feature maps at different stages.

![CNN feature maps at different layers](https://www.researchgate.net/profile/Alexey-Dosovitskiy/publication/336982709/figure/fig1/AS:819396583350274@1572427898200/Overview-of-a-convolutional-neural-network-CNN.png)

*Reference: [CNN architecture and feature extraction](https://cs.brown.edu/people/pfelzens/teaching/vision/)*

### Early CNN layers:

- Preserve more spatial details.
- Detect edges, textures and simple patterns.
- Have higher spatial resolution.
- Are useful for retaining small-object details.

### Deeper CNN layers:

- Capture more semantic information.
- Recognize object parts and high-level patterns.
- Have lower spatial resolution.
- Often provide stronger context for larger objects.

For example, consider a CNN processing a 640 × 640 image.

| CNN stage | Example feature map | Characteristics |
|---|---:|---|
| Early layer | 160 × 160 | High spatial detail |
| Middle layer | 80 × 80 | Intermediate features |
| Deeper layer | 40 × 40 | More semantic information |
| Deepest layer | 20 × 20 | Strong high-level features |

*Illustrative dimensions for a CNN with progressively downsampled feature maps.*

The challenge is combining detailed spatial information with high-level semantic information.

That's where Feature Pyramids become useful.

## 1.3 What is a Feature Pyramid?

A **Feature Pyramid** is a collection of feature maps at different spatial resolutions that helps an object detector recognize objects of different sizes.

![Feature Pyramid Network architecture](https://miro.medium.com/v2/resize%3Afit%3A1400/1%2AcHR4YRqdPBOx4IDqzU-GwQ.png)

*Reference: [Jonathan Hui — Feature Pyramids and Object Detection](https://jonathan-hui.medium.com/what-do-we-learn-from-single-shot-object-detectors-ssd-yolov3-fpn-focal-loss-3888677c5f4d)*

One important architecture is the **Feature Pyramid Network (FPN)**, introduced in 2017.

FPN combines features from different CNN stages using:

1. Bottom-up pathway
2. Top-down pathway
3. Lateral connections

Let's understand each.

### A. Bottom-up pathway

The input image passes through the CNN.

As it moves deeper:

- Spatial resolution decreases.
- Semantic information generally increases.

```text
       Input Image
            |
            ▼
       C2 ── High resolution
            |
            ▼
           C3
            |
            ▼
           C4
            |
            ▼
       C5 ── Low resolution
```

### B. Top-down pathway

High-level semantic features are passed back toward higher-resolution feature maps through upsampling.

### C. Lateral connections

Features from corresponding CNN stages are combined with the upsampled deeper features.

This helps combine:

- Fine spatial details.
- Strong semantic information.

## 1.4 FPN Architecture

![Feature Pyramid Network with bottom-up, top-down and lateral connections](https://miro.medium.com/v2/resize%3Afit%3A1400/1%2AcHR4YRqdPBOx4IDqzU-GwQ.png)

*Reference: [Feature Pyramid Network architecture](https://jonathan-hui.medium.com/what-do-we-learn-from-single-shot-object-detectors-ssd-yolov3-fpn-focal-loss-3888677c5f4d)*

Conceptually:

```text
                  Input Image
                       |
                       ▼
                   CNN Backbone
                       |
                ┌──────┼──────┐
                ▼      ▼      ▼
               C3     C4     C5
                |      |      |
                └──────┴──────┘
                       |
                       ▼
                Top-Down Pathway
                       |
                       ▼
                Lateral Connections
                       |
                ┌──────┼──────┐
                ▼      ▼      ▼
               P3     P4     P5
                |      |      |
                ▼      ▼      ▼
             Small  Medium  Large
             Objects Objects Objects
```

The resulting pyramid features can be used by detection heads to make predictions at different scales.

**Important:** FPN is a feature-extraction and feature-fusion design. It is not a complete object detector by itself.

## 1.5 Where are Feature Pyramids used?

- Faster R-CNN with FPN.
- RetinaNet.
- Many YOLO-style architectures use multi-scale feature fusion, although their neck designs may differ from the original FPN.

### What you should remember:

Feature pyramids help object detectors handle objects of different sizes by combining high-resolution spatial information with deeper semantic information.

---

# 2. Anchor-Based vs Anchor-Free Detection

<badge>Must Know</badge>

This is one of the most important differences in modern object detection.

Before understanding anchor-free detection, let's recall anchor boxes from Faster R-CNN and SSD.

## 2.1 Anchor-Based Detection

An **anchor box** is a predefined reference bounding box placed at a particular location in a feature map.

These boxes have different:

- Sizes.
- Aspect ratios.

![Anchor boxes with different scales and aspect ratios](https://storage.googleapis.com/omelet-f0b89.appspot.com/public/blog/yolov8-deep-dive-anchor-free-dfl/anchor-free.png)

*Reference: [Anchor-Based vs Anchor-Free Detection](https://www.omelet.tech/yolov8-deep-dive-anchor-free-dfl/)*

For example, an object detector might use:

| Anchor | Shape |
|---|---|
| Anchor A | Small square |
| Anchor B | Large square |
| Anchor C | Wide rectangle |
| Anchor D | Tall rectangle |

The detector predicts:

1. Whether an anchor contains an object.
2. Which object class it represents.
3. How to adjust its coordinates.

## 2.2 Anchor-Based Prediction Pipeline

```text
       CNN Feature Map
              |
              ▼
       Generate Anchor Boxes
              |
              ▼
        Object Classification
              |
              ▼
       Bounding Box Regression
              |
              ▼
          Refined Boxes
              |
              ▼
             NMS
              |
              ▼
        Final Detections
```

### Example:

Suppose an anchor box is:

$$
(100,100,200,200)
$$

The model predicts coordinate adjustments to move and resize this reference box to better match the actual object.

This is called **bounding-box regression**.

## 2.3 Limitations of Anchor-Based Detection

Anchor-based detectors have some challenges:

- Choosing suitable anchor sizes and aspect ratios.
- Handling objects with unusual shapes.
- Managing large numbers of candidate anchors.
- Assigning anchors to ground-truth objects during training.
- Dealing with foreground-background imbalance.

These challenges motivated anchor-free approaches.

## 2.4 Anchor-Free Detection

<badge color="success">Important Modern Concept</badge>

Anchor-free detectors predict object locations without requiring predefined anchor boxes.

Instead of starting with multiple reference boxes at each location, the model directly predicts object properties.

![Anchor-free object detection illustration](https://storage.googleapis.com/omelet-f0b89.appspot.com/public/blog/yolov8-deep-dive-anchor-free-dfl/anchor-free.png)

*Reference: [Omelet Tech — YOLOv8 Anchor-Free Detection](https://www.omelet.tech/yolov8-deep-dive-anchor-free-dfl/)*

Depending on the architecture, an anchor-free detector may predict:

- Object centre points.
- Bounding-box boundaries.
- Distances from a reference point to box boundaries.
- Object classification scores.

### Example: Centre-Based Prediction

Imagine an object detector identifying a car.

Instead of matching the car to a predefined anchor box, the model predicts a centre point and the distances to its boundaries.

![Anchor-free centre-point and bounding-box prediction](https://storage.googleapis.com/omelet-f0b89.appspot.com/public/blog/yolov8-deep-dive-anchor-free-dfl/anchor-free.png)

*Reference: [Anchor-free prediction concept](https://www.omelet.tech/yolov8-deep-dive-anchor-free-dfl/)*

Suppose the model predicts:

- Centre X = 200
- Centre Y = 150
- Left distance = 60
- Top distance = 40
- Right distance = 80
- Bottom distance = 50

Then:

$$
x_1 = 200-60=140
$$

$$
y_1 = 150-40=110
$$

$$
x_2 = 200+80=280
$$

$$
y_2 = 150+50=200
$$

The resulting bounding box is:

$$
(140,110,280,200)
$$

This is one example of an anchor-free prediction strategy. Different anchor-free architectures use different representations.

## 2.5 Anchor-Based vs Anchor-Free Comparison

| Feature | Anchor-Based | Anchor-Free |
|---|---|---|
| Predefined anchor boxes | Yes | No |
| Prediction | Adjust anchors | Direct object geometry or locations |
| Anchor configuration | Required | Not required |
| Candidate assignment | Anchor matching | Architecture-specific point/region assignment |
| Example | Original Faster R-CNN, SSD | FCOS, CenterNet, YOLOv8-style detection |

**Important:** Anchor-free does not mean the model has no candidate locations or feature-map grids. It means it does not rely on predefined anchor boxes as its reference boxes.

Also, YOLO is a family, not one fixed design. Different YOLO generations use different detection mechanisms.

---

# 3. Transformer-Based Object Detection

<badge>Must Know Conceptually</badge>

Until now, most of our object detection architectures have been based on CNNs.

CNNs are particularly good at learning local visual patterns:

- Edges.
- Textures.
- Shapes.
- Object parts.

But there is another powerful architecture that can model relationships across different parts of an image.

**Transformers.**

You have already studied Transformers in your NLP curriculum.

Remember concepts such as:

- Attention.
- Self-attention.
- Multi-head attention.
- Positional encoding.
- Encoder and decoder.

These concepts can also be applied to images.

## 3.1 Why use Transformers in Computer Vision?

Imagine an image containing:

- A person riding a bicycle.
- A car behind the bicycle.
- A traffic signal in the background.

Understanding the image may require considering relationships between objects and their surroundings.

For example:

- Which person is riding the bicycle?
- Where is the bicycle relative to the person?
- Which visual regions belong to the same object?

Self-attention helps a model relate information from different spatial locations.

![Transformer self-attention across image patches](https://miro.medium.com/v2/resize%3Afit%3A1400/1%2AlIkgUR0ge8QzMrA04RxNFw.png)

*Reference: [Vision Transformer — Self-Attention](https://medium.com/%40sumanthvinnu14/how-vision-transformers-work-an-intuitive-visual-deep-dive-0575ae7d6c61)*

## 3.2 How does a Transformer process an image?

A typical vision transformer pipeline:

```text
          Input Image
               |
               ▼
          Divide into Patches
               |
               ▼
         Convert Patches
         into Embeddings
               |
               ▼
        Add Positional Information
               |
               ▼
          Transformer Encoder
               |
               ▼
          Contextual Features
```

Let's understand the stages.

### Step 1 — Image Patching

The image is divided into smaller patches.

For example, a 224 × 224 image can be divided into 16 × 16 patches.

Number of patches:

$$
N=\frac{224}{16}\times\frac{224}{16}
$$

$$
N=14\times14=196
$$

![Vision Transformer image patching](https://substackcdn.com/image/fetch/%24s_%21_InE%21%2Cw_1200%2Ch_675%2Cc_fill%2Cf_jpg%2Cq_auto%3Agood%2Cfl_progressive%3Asteep%2Cg_auto/https%3A/substack-post-media.s3.amazonaws.com/public/images/32d56720-f275-424c-be31-54b7d02d5fdb_1316x688.png)

*Reference: [Vision Transformer pipeline illustration](https://togoailabs.substack.com/p/the-fall-of-cnns)*

### Step 2 — Patch Embeddings

Each patch is converted into a numerical vector.

These vectors represent the image patches in an embedding space.

### Step 3 — Positional Information

Transformers need information about the spatial position of patches.

Positional encodings or embeddings provide this spatial information.

### Step 4 — Self-Attention

The transformer uses self-attention to learn relationships between patch representations.

For example, a patch containing a car wheel can relate to other patches containing the car body.

![Multi-head self-attention mechanism](https://miro.medium.com/v2/resize%3Afit%3A1400/1%2AlIkgUR0ge8QzMrA04RxNFw.png)

*Reference: [Vision Transformer self-attention](https://medium.com/%40sumanthvinnu14/how-vision-transformers-work-an-intuitive-visual-deep-dive-0575ae7d6c61)*

### Step 5 — Contextual Features

The output contains contextualized representations that can be used for downstream computer vision tasks, including object detection.

## 3.3 CNN vs Transformer in Computer Vision

| Feature | CNN | Transformer |
|---|---|---|
| Main mechanism | Convolution | Self-attention |
| Information processing | Local receptive fields that grow through layers | Relationships between tokens |
| Spatial information | Built into convolution and architecture | Often provided through positional information |
| Long-range relationships | Built progressively | Can model directly through attention |
| Example | ResNet | ViT |

**Important:** This is a conceptual comparison. Modern CNNs can also capture broad context, and modern vision transformers often use hierarchical or local attention rather than unrestricted global attention.

---

# 4. DETR — Detection Transformer

<badge color="success">Must Know</badge>

![DETR architecture overview](https://miro.medium.com/v2/resize%3Afit%3A1358/format%3Awebp/1%2AamkuiFy6dkb49xYAvABfvg.png)

*Reference: [DETR architecture illustration](https://medium.com/%40mithilmaske/fine-tuning-owl-vit-on-a-custom-object-detection-dataset-5b1947c21269)*

DETR stands for **DEtection TRansformer**.

It was introduced by Facebook AI Research (now Meta AI) in 2020.

DETR brought Transformer-based object detection into a prominent end-to-end detection architecture.

Its central idea:

> Treat object detection as a direct set-prediction problem using a Transformer.

This is an important shift from traditional detection pipelines.

## 4.1 What is different about DETR?

Traditional detectors often involve:

- Anchor boxes or candidate locations.
- Region proposals in two-stage models.
- Bounding-box regression.
- NMS to remove duplicate detections.

Original DETR uses a different approach.

It predicts a fixed-size set of object predictions and trains the model to match these predictions to the actual ground-truth objects.

It uses:

- CNN backbone.
- Transformer encoder.
- Transformer decoder.
- Object queries.
- Prediction heads.
- Bipartite matching.

Let's understand each component.

## 4.2 DETR Architecture

![DETR end-to-end architecture](https://miro.medium.com/v2/resize%3Afit%3A1358/format%3Awebp/1%2AamkuiFy6dkb49xYAvABfvg.png)

*Reference: [DETR architecture](https://medium.com/%40mithilmaske/fine-tuning-owl-vit-on-a-custom-object-detection-dataset-5b1947c21269)*

```text
                Input Image
                     |
                     ▼
                CNN Backbone
                     |
                     ▼
                Feature Map
                     |
                     ▼
             Flatten Feature Map
                     |
                     ▼
             Positional Encoding
                     |
                     ▼
             Transformer Encoder
                     |
                     ▼
             Transformer Decoder
                     ▲
                     |
                Object Queries
                     |
                     ▼
               Prediction Heads
                 /         \
                ▼           ▼
          Classification  Bounding Box
                |           |
                └─────┬─────┘
                      ▼
               Final Predictions
```

Let's break it down.

## 4.3 CNN Backbone

The input image passes through a CNN backbone such as ResNet.

The backbone extracts visual features.

![CNN backbone feature extraction](https://www.researchgate.net/profile/Alexey-Dosovitskiy/publication/336982709/figure/fig1/AS:819396583350274@1572427898200/Overview-of-a-convolutional-neural-network-CNN.png)

## 4.4 Transformer Encoder

The CNN feature map is converted into a sequence of feature representations.

Positional information is added.

The Transformer encoder applies self-attention to capture relationships across the image features.

## 4.5 Object Queries

<badge color="success">Very Important Concept</badge>

Object queries are learned embeddings supplied to the Transformer decoder.

Think of them as a fixed number of slots that can produce object predictions.

For example, suppose DETR is configured with 100 object queries.

The decoder receives 100 query embeddings.

Each query can produce:

- A predicted object class.
- A predicted bounding box.

![DETR object queries and Transformer decoder](https://substackcdn.com/image/fetch/%24s_%21UAEg%21%2Cf_auto%2Cq_auto%3Agood%2Cfl_progressive%3Asteep/https%3A/substack-post-media.s3.amazonaws.com/public/images/de08d6bf-5025-426d-a22b-1c6fc5cdada6_1149x1122.png)

*Reference: [DETR decoder and object queries](https://www.vizuaranewsletter.com/p/detection-transformer-detr-an-introduction)*

Imagine an image containing only three objects.

The model might produce:

```text
Query 1  → Car
Query 2  → Pedestrian
Query 3  → Bicycle

Query 4  → No object
Query 5  → No object
...
Query 100 → No object
```

The unused queries predict a special no-object class.

This is how DETR can use a fixed number of prediction slots even when images contain different numbers of objects.

## 4.6 Prediction Heads

Each decoder output is passed through prediction heads.

### Classification head:

Predicts an object class or the no-object class.

### Bounding box head:

Predicts normalized bounding-box coordinates.

For example:

```text
Class       → Car
Confidence  → 0.95

Bounding Box:
x_center = 0.50
y_center = 0.45
width    = 0.30
height   = 0.25
```

These are normalized coordinates, not pixel coordinates.

---

# 5. Hungarian Matching — How DETR Learns

<badge color="success">Important</badge>

This is one of the most interesting concepts in DETR.

Traditional detectors often use anchor assignment or proposal matching during training.

DETR instead uses **bipartite matching** to match predicted object slots to ground-truth objects.

The original DETR uses the Hungarian algorithm to find an optimal one-to-one matching.

## Example

Suppose the ground truth contains three objects:

**Ground Truth:**

1. Car
2. Person
3. Bicycle

The model generates five predictions:

**Predictions:**

1. Car
2. Bicycle
3. Car
4. Person
5. No Object

![Hungarian matching concept in object detection](https://substackcdn.com/image/fetch/%24s_%21UAEg%21%2Cf_auto%2Cq_auto%3Agood%2Cfl_progressive%3Asteep/https%3A/substack-post-media.s3.amazonaws.com/public/images/de08d6bf-5025-426d-a22b-1c6fc5cdada6_1149x1122.png)

*Reference: [DETR matching and prediction process](https://www.vizuaranewsletter.com/p/detection-transformer-detr-an-introduction)*

Conceptually:

| Ground truth | Matched prediction |
|---|---|
| Car | Prediction 1 |
| Person | Prediction 4 |
| Bicycle | Prediction 2 |

The remaining predictions are trained as no-object predictions.

The matching cost considers classification and bounding-box accuracy, including box distance and overlap.

### Why is this important?

It encourages the model to produce a unique prediction for each ground-truth object instead of generating many duplicate predictions.

## 5.1 Why Original DETR Does Not Need Traditional NMS

Remember NMS from Module 40?

Traditional detectors can produce multiple overlapping predictions for the same object, so NMS removes duplicates.

Original DETR uses set prediction with one-to-one matching during training.

This encourages unique predictions, so its standard inference pipeline does not require traditional NMS.

![DETR set prediction and object queries](https://miro.medium.com/v2/resize%3Afit%3A1358/format%3Awebp/1%2AamkuiFy6dkb49xYAvABfvg.png)

**Important:** This is a property of original DETR's set-prediction approach, not a guarantee that every Transformer-based detector is NMS-free.

---

# 6. Limitations of Original DETR

DETR introduced an important architectural change, but the original implementation had some challenges:

1. **Slow convergence:** It generally required many training epochs to reach strong performance.
2. **Small object detection:** The original feature resolution and attention design made detecting small objects challenging.
3. **Computational cost:** Global self-attention can be expensive for large feature sequences.

Later DETR variants addressed these limitations through architectural and training improvements.

Examples include:

- Deformable DETR.
- Conditional DETR.
- DAB-DETR.
- DINO.

![DETR Transformer architecture](https://miro.medium.com/v2/resize%3Afit%3A1358/format%3Awebp/1%2AamkuiFy6dkb49xYAvABfvg.png)

For now, understand the original DETR architecture conceptually. We don't need to derive every variant's mathematical details.

---

# 7. Complete Comparison: Traditional vs Modern Detection

| Architecture / concept | Main contribution |
|---|---|
| Faster R-CNN | Learned region proposals using RPN |
| SSD | Multi-scale detection using multiple feature maps |
| RetinaNet | Focal Loss for foreground-background imbalance |
| YOLO | Efficient direct object detection |
| FPN | Combines features across different resolutions |
| Anchor-free detection | Predicts object geometry without predefined anchor boxes |
| DETR | Transformer-based set prediction with object queries |

## How the concepts connect

![Feature Pyramid Network multi-scale detection illustration](https://miro.medium.com/v2/resize%3Afit%3A1400/1%2AcHR4YRqdPBOx4IDqzU-GwQ.png)

Think of modern object detection as three major developments:

1. **Better visual representations:** Feature pyramids and multi-scale feature fusion.
2. **Better box prediction strategies:** Anchor-based and anchor-free detection.
3. **Attention-based detection:** Transformers and DETR-style set prediction.

---

# 8. Must Know vs Good to Know

| Priority | Concepts |
|---|---|
| Must Know | Feature Pyramid Network and why it is needed |
| Must Know | Anchor-based vs anchor-free |
| Must Know | Transformer self-attention in vision |
| Must Know | DETR architecture |
| Must Know | Object queries and Hungarian matching |
| Good to Know | FPN top-down and lateral connections |
| Good to Know | DETR convergence limitations |
| Optional | Mathematical derivation of Hungarian matching and deformable attention |

---



# What You Should Remember

1. **Feature Pyramids:** Combine different spatial resolutions and semantic features to detect small, medium and large objects.
2. **Anchor-based:** Uses predefined reference boxes and predicts adjustments.
3. **Anchor-free:** Predicts object locations or bounding-box geometry without predefined anchor boxes.
4. **Transformers:** Use self-attention to model relationships between image features.
5. **DETR:** Uses a CNN backbone, Transformer encoder-decoder and object queries.
6. **Hungarian Matching:** Performs one-to-one matching between predictions and ground-truth objects.
7. **NMS:** Original DETR does not require traditional NMS in its standard inference pipeline.

---